# Référence
Ce TP est inspiré de l'excellent livre de François Chollet, "Deep Learning with Python". 

## Mise en route

### Environnement Python au CREMI

Un environnement `arna-nlp` contenant toutes les dépendances du TP est déjà
installé au CREMI. Pour l'activer dans un terminal :

```sh
source ~/espaces/ens/arna-nlp/bin/activate
```

Pour l'utiliser comme noyau Jupyter, lancez une seule fois, avec
l'environnement activé :

```sh
python -m ipykernel install --user --name=arna-nlp
```

Ensuite, il suffit d'ouvrir le notebook dans Jupyter et de choisir le noyau
`arna-nlp`, sans avoir besoin de réactiver l'environnement à chaque fois.

### Données

Ce notebook a besoin de deux fichiers de données, qui ne sont pas dans le dépôt
(ils sont trop volumineux). Téléchargez-les depuis la page **Moodle** du cours :

- `IMDB Dataset.csv`
- `glove.6B.100d.txt`

Rangez-les à côté du notebook, avec GloVe dans un sous-dossier `glove/` :

```
nlp.ipynb
IMDB Dataset.csv
glove/
  glove.6B.100d.txt
```

Vous pouvez aussi les ranger dans le dossier de votre choix (toujours avec le
sous-dossier `glove/`), en définissant la variable d'environnement `ARNA_DATA`
vers ce dossier. La cellule suivante
vérifie que les deux fichiers sont bien trouvés.

In [1]:
# Dépendances (décommentez si besoin, par exemple sur Colab)
# %pip install -q keras tensorflow pandas numpy sentence-transformers

import os
from pathlib import Path

import keras
import numpy as np

# Dossier contenant les données. Par défaut : à côté du notebook.
DATA_DIR = Path(os.environ.get("ARNA_DATA", "."))

PATH_IMDB = DATA_DIR / "IMDB Dataset.csv"
PATH_GLOVE = DATA_DIR / "glove" / "glove.6B.100d.txt"

for name, path in [("IMDB", PATH_IMDB), ("GloVe", PATH_GLOVE)]:
    print(f"{name:6s} {'OK ' if path.exists() else 'MANQUANT'}  {path}")

I0000 00:00:1791376525.835503   95916 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791376529.088596   95916 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791376541.794859   95916 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


IMDB   OK   IMDB Dataset.csv
GloVe  OK   glove/glove.6B.100d.txt


# Préparer les données textuelles pour le machine learning

Les modèles de machine learning ne peuvent pas traiter les chaînes de caractères directement. Nous devons les convertir en tenseurs de nombres. La manière classique de procéder est la suivante :

- Standardiser le texte - par exemple, en mettant tout en minuscules, en supprimant la ponctuation, etc.
- Tokeniser le texte - c'est-à-dire, en le diviser en caractères, mots ($1$-gramme), groupes de mots ($N$-grammes), etc.
- Indexer les tokens présents dans le texte
- Convertir les tokens en vecteurs de nombres

## Indexer les tokens via TextVectorization

In [2]:
# Une des façons de procéder est d'utiliser le layer TextVectorization de Keras. 
#Par défaut, ce layer convertit en lowercase, retire la ponctuation et sépare les mots par des espaces, 
#mais ceci est hautement personnalisable.

text_vectorization = keras.layers.TextVectorization(output_mode="int")

E0000 00:00:1791376615.455761   95916 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1791376615.994062   95916 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
dataset = [
    "Je suis un texte",
    "Je suis Un autre texte",
    "Je suis un autre autre texte",
    "Je ne suis pas un chat",
    "Si je pense, je suis",
    "J'écris donc je suis",
    "Un écrivain sait écrire",
]

text_vectorization.adapt(dataset)

In [4]:
vocabulary = text_vectorization.get_vocabulary()
print(vocabulary)

['', '[UNK]', np.str_('je'), np.str_('suis'), np.str_('un'), np.str_('texte'), np.str_('autre'), np.str_('écrivain'), np.str_('écrire'), np.str_('si'), np.str_('sait'), np.str_('pense'), np.str_('pas'), np.str_('ne'), np.str_('jécris'), np.str_('donc'), np.str_('chat')]


#### Questions :

- Que constatez-vous en examinant le vocabulaire généré ?
- Comment les mots sont-ils triés ? Pourquoi certains mots apparaissent-ils en premier ?

In [5]:
#Pas de doublons, les mots les plus apparus apparaissent en premier.
# On peut encoder un texte sous forme de séquence d'entiers.

test = "Si je sais écrire alors je ne suis ni un chat ni une souris."
encoded_text = text_vectorization(test)
print(encoded_text)

tf.Tensor([ 9  2  1  8  1  2 13  3  1  4 16  1  1  1], shape=(14,), dtype=int64)


In [24]:
# Pour le décoder, on fait l'opération inverse. Ici on remarque que certain token sont inconnus, car ils n'apparaissent pas dans le dataset de départ. C'est le token [UNK] d'indice 1 qui est réservé pour les mots inconnus.

#TODO: Décoder le texte encodé
# Votre code ici
decoded_words = []
print(len(encoded_text))
for i in range(len(encoded_text)):
    word = "".join(vocabulary[encoded_text.numpy()[i]])
    decoded_words.append(word)
    
decoded_text = "".join(decoded_words)
print(decoded_text)




14
sije[UNK]écrire[UNK]jenesuis[UNK]unchat[UNK][UNK][UNK]


#### Questions :

- Quels mots sont transformés en token [UNK] (inconnu) ?
- Pourquoi ces mots sont-ils inconnus ? Que pouvez-vous faire pour les ajouter au vocabulaire ?

### Remarques :
- `TextVectorization` est un layer Keras. Cependant, c'est essentiellement un dictionnaire qui ne peut pas être exécuté sur un GPU ou un TPU. Si vous avez besoin de performances, il est recommandé de l'utiliser en dehors du modèle, en tant qu'étape de prétraitement dans le pipeline de données, sinon il deviendra un facteur limitant de la performance du modèle.

- **Padding** : si vous avez des séquences de longueurs différentes, vous pouvez les tronquer ou les remplir pour qu'elles aient toutes la même longueur. C'est ce que fait le paramètre `output_sequence_length`.

In [30]:
text_vectorization = keras.layers.TextVectorization(output_mode="int", output_sequence_length=10)

dataset = [
    ["Texte court, je vais être paddé"],
    ["Je suis un texte trop long donc je vais être tronqué. C'est triste."],
]

text_vectorization.adapt(dataset)

print(text_vectorization.get_vocabulary())

print(text_vectorization(dataset))


['', '[UNK]', np.str_('je'), np.str_('être'), np.str_('vais'), np.str_('texte'), np.str_('un'), np.str_('trop'), np.str_('tronqué'), np.str_('triste'), np.str_('suis'), np.str_('paddé'), np.str_('long'), np.str_('donc'), np.str_('court'), np.str_('cest')]
tf.Tensor(
[[ 5 14  2  4  3 11  0  0  0  0]
 [ 2 10  6  5  7 12 13  2  4  3]], shape=(2, 10), dtype=int64)


#### Questions :

- Que se passe-t-il si la séquence est plus longue que la longueur définie ?
- Que se passe-t-il si elle est plus courte ?

## Convertir les tokens avec un one-hot encoding

In [31]:
# On peut utiliser un layer de type CategoryEncoding pour convertir une séquence d'entiers 
#en une séquence de vecteurs one-hot

one_hot_layer = keras.layers.CategoryEncoding(num_tokens=len(text_vectorization.get_vocabulary()), output_mode="one_hot")
text = "Je suis un texte très court"
one_hot_layer(text_vectorization(text))

<tf.Tensor: shape=(10, 16), dtype=float32, numpy=
array([[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],
      dtype=float32)>

#### Questions :

- Que représente chaque vecteur produit par le `one-hot encoding` ?
- Quelles sont les limitations de cette méthode ? Quantifier ces limitations avec un exemple.

## N-grams et représentation des phrases sous forme "bag of words"

- On peut également représenter une phrase comme un ensemble de tokens non ordonnés. Afin de capturer un peu de contexte, on utilise souvent des n-grams. Par exemple, si n=2, alors "je suis un texte" deviendrait "je", "suis", "un", "texte", "je suis", "suis un", "un texte". Regardez à quoi ressemble le vocabulaire après avoir utilisé un 2-gram par exemple.

- `multi_hot` : permet de représenter les phrases sous forme de "bag of words". Chaque mot est représenté par un vecteur de 0 et de 1. Si le token est présent dans la phrase, alors la valeur correspondante est 1, sinon 0.

In [32]:
# Attention : on repart du petit corpus du début, et non du corpus tronqué
# de la cellule précédente.
corpus = [
    "Je suis un texte",
    "Je suis Un autre texte",
    "Je suis un autre autre texte",
    "Je ne suis pas un chat",
    "Si je pense, je suis",
    "J'écris donc je suis",
    "Un écrivain sait écrire",
]

text_vectorization = keras.layers.TextVectorization(
    max_tokens=1000,
    ngrams=2,
    output_mode="multi_hot"
)

text_vectorization.adapt(corpus)

vocabulary = text_vectorization.get_vocabulary()
inverse_dictionary = {idx: word for idx, word in enumerate(vocabulary)}
print(inverse_dictionary)

{0: '[UNK]', 1: np.str_('je'), 2: np.str_('suis'), 3: np.str_('un'), 4: np.str_('je suis'), 5: np.str_('texte'), 6: np.str_('suis un'), 7: np.str_('autre'), 8: np.str_('un autre'), 9: np.str_('autre texte'), 10: np.str_('écrivain sait'), 11: np.str_('écrivain'), 12: np.str_('écrire'), 13: np.str_('un écrivain'), 14: np.str_('un texte'), 15: np.str_('un chat'), 16: np.str_('suis pas'), 17: np.str_('si je'), 18: np.str_('si'), 19: np.str_('sait écrire'), 20: np.str_('sait'), 21: np.str_('pense je'), 22: np.str_('pense'), 23: np.str_('pas un'), 24: np.str_('pas'), 25: np.str_('ne suis'), 26: np.str_('ne'), 27: np.str_('jécris donc'), 28: np.str_('jécris'), 29: np.str_('je pense'), 30: np.str_('je ne'), 31: np.str_('donc je'), 32: np.str_('donc'), 33: np.str_('chat'), 34: np.str_('autre autre')}


In [33]:
# Avec multi_hot, on est dans le setting "bag of words", où un texte est représenté par
#un vecteur de la taille du vocabulaire, 
#où chaque élément vaut 0 ou 1 selon que le token correspondant est présent ou non dans le texte. 

text = "Je suis un texte très court"
encoded_text = text_vectorization(text)
print(encoded_text)

tf.Tensor([1 1 1 1 1 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0], shape=(35,), dtype=int64)


#### Questions :

- Comment les bigrammes influencent-ils la taille du vocabulaire ?
- En quoi cette représentation diffère-t-elle du one-hot encoding ?
- Est-ce que l'on peut encore décoder les vecteurs multi-hot pour retrouver les phrases d'origine ?


# IMDB dataset

On utilise le fichier .csv "IMDB Dataset.csv" qui contient 50 000 critiques de films, chacune étant étiquetée comme "positive" ou "negative". Ce fichier provient de Kaggle : https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
- Vous pouvez soit télécharger directement le csv soit utiliser kagglehub (le code correspondant est donnée sur kaggle lorsque l'on clique sur download)
- Pour les détails sur le dataset : https://ai.stanford.edu/~amaas/data/sentiment/


In [34]:
import pandas as pd

df = pd.read_csv(PATH_IMDB)

df["label"] = (df["sentiment"] == "positive").astype(int)

df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [35]:
# print a small review
N = 3
print(df["review"].iloc[N])
print(df["label"].iloc[N])

Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.
0


In [36]:
# Split des données en train et test
# On utilise 80% des données pour l'entraînement et 20% pour le test.

max_tokens = 10_000

# shuffle the data
df = df.sample(frac=1.0, random_state=0)

# split the data
train_size = int(len(df) * 0.8)
train_df = df[:train_size]
test_df = df[train_size:]

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

# preprocessing les données avec TextVectorization, multi_hot

text_vectorization = keras.layers.TextVectorization(max_tokens=max_tokens, output_mode="multi_hot")
# On n'adapte que sur le train : le vocabulaire ne doit pas
# dépendre des données de test.
text_vectorization.adapt(train_df["review"])

train_vectorized = text_vectorization(train_df["review"])
test_vectorized = text_vectorization(test_df["review"])

Train size: 40000
Test size: 10000


In [54]:
train_vectorized.shape

TensorShape([40000, 300])

In [55]:
train_vectorized[0]

<tf.Tensor: shape=(300,), dtype=int64, numpy=
array([  314,  5290,     7,    21,     2,   508,    37,     2,  1381,
          28,     7,    31,     2,  1309,     5,     2,  7732,    28,
        1039, 18287,  3395,  9262,    15,    25,  3208,     4,    35,
         931,   415,   332, 11764,     3,   419,     5,  9111,   442,
           3,  5810,     5,     3, 15559,   246,    33,   699,     1,
          13,    15,   283,    36,    45,   946,     8,     3,  7352,
         533,    51,    10,    14,  1078,    10,    69,   364,    23,
          12,     2,  7352,  1885,   590,    16, 16134, 12132,  3048,
           4, 18449,     1,  8345,    24,    15,   791,    15,     9,
         207,   194,    75,     8,  1885,    39,   128,   533,   253,
          56,    29,   186, 15896,    13,     2,  2868,     5,  3395,
           1,  4801,     7,     3,  6585,    21,  4892,  7700,    82,
          10,    22,  7469,    11,     7,    34,  8590,  2703,    20,
          63,    31,     2,    60,    14,   

In [57]:
# TODO: Créer un modèle simple avec Keras pour la classification de texte.
# - un input layer
# - une couche dense de 16 neurones avec une activation relu
# - une couche dense de 1 neurone avec une activation sigmoid
from keras import layers

model = keras.models.Sequential([
    layers.Input(shape=(10000,)),
    layers.Dense(2, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(train_vectorized, train_df["label"], epochs=10, batch_size=32, validation_split=0.2)

print(f"Evaluation du modèle: {model.evaluate(test_vectorized, test_df['label'])[1]:.3f}")



Epoch 1/10


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense_3' is incompatible with the layer: expected axis -1 of input shape to have value 10000, but received input with shape (32, 300)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 300), dtype=int64)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [45]:
# Regardons les prédictions du modèle sur un sample tiré du test set

sample = test_df.sample(1)
print(sample["review"].values[0])
print(f"Label: {sample['label'].values[0]}")
print(f"Prédiction: {model.predict(text_vectorization(sample['review']))[0][0]:.3f}")


I have just managed to get hold of the Celestial region 3 DVD of The Five Venoms and what a super job they have made of it. A fantastic digitally remastered transfer and a must have for any Kung Fu fan.<br /><br />The story is pretty straight forward, and has been mentioned already so I won't go into it again. Needless to say it's the fight scenes that many buy the movie for, and they do not disappoint. Only problem is they are a bit few and far between and seem over rehearsed. Bruce Lee could take these lot on and drink a cup of tea at the same time! All kicks and punches come with the all important "type writer" clicks and air "whooshes", which is a cool effect though quite amusing!<br /><br />I give this movie a B+. Good but not great. In a way I feel it could have been that bit better. Golden Swallow, by the same director a decade earlier, had as good a story and better action.
Label: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Prédiction: 0.715


Est-ce que les prédictions de votre modèle semblent pertinentes ?

# La manière séquentielle de traiter les phrases en utilisant des **word embeddings**

**Problème :**  
Le `one-hot encoding` est une méthode inefficace pour représenter des mots, car :

- Il ne capture pas les relations entre les mots (par exemple, "chat" et "chien" sont totalement indépendants alors qu'ils sont sémantiquement proches).
- Il crée des vecteurs de très grande dimension (un vocabulaire de 20 000 mots pour une phrase de 600 mots donnerait un vecteur de 600 x 20 000 = 12 000 000 de dimensions, ce qui est beaucoup trop lourd).

L'objectif est de représenter une séquence de $k$ mots sous forme d'un tenseur de dimension $k \times d$, où $d$ est la dimension de l'embedding des mots, avec $d$ relativement petit (de l'ordre de la centaine). Ce tenseur peut ensuite être passé à un modèle séquentiel tel qu'un **RNN**, **LSTM**, **GRU**, ou **Transformer**.


La question est donc : comment obtenir ces embeddings ? Il y a deux manières de procéder :
- On peut entraîner nos propres embeddings, en les initialisant aléatoirement, puis en les ajustant pour minimiser une fonction de coût par backpropagation. Pour cela, on utilise le layer `Embedding` de Keras.
- On peut utiliser des embeddings de mots pré-entraînés, qui ont été entraînés sur un très grand corpus de texte, et qui capturent donc des relations sémantiques relativement générique entre les mots.


#### Étape 1 : Etape d'indexation du texte avec un modèle séquentiel

- On commence par indexer les tokens avec `TextVectorization` comme précédemment. On doit obtenir des listes de tokens, où chaque token est représenté par un entier.
- Mettez une limite de 20 000 tokens dans le vocabulaire, et une longueur maximale de 300 mots.

In [46]:
max_tokens = 20_000
text_vectorization = keras.layers.TextVectorization(max_tokens=max_tokens, output_mode="int", output_sequence_length=300)

text_vectorization.adapt(train_df["review"])

train_vectorized = text_vectorization(train_df["review"])
test_vectorized = text_vectorization(test_df["review"])

print(train_vectorized.shape)


(40000, 300)


#### Étape 2 : Création d'une couche d'embedding et d'un modèle LSTM

- Créez un modèle LSTM qui intègre la couche `Embedding` pour générer des représentations denses de dimension 256. Votre modèle doit avoir un layer input, un layer embedding, un layer LSTM, et un layer dense avec une activation sigmoid.
- Entrainez le modèle sur les données d'entraînement et évaluez-le sur les données de test.

In [52]:
input_layer = keras.layers.Input(shape=(300,))
embedding_layer = keras.layers.Embedding(20000,256)  # TODO: keras.layers.Embedding(...)
lstm_layer = keras.layers.LSTM(256)
dense_layer = layers.Dense(1, activation="sigmoid")

model = keras.models.Sequential([
    input_layer,
    embedding_layer,
    lstm_layer,
    dense_layer
    
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(train_vectorized, train_df["label"], epochs=4, batch_size=32, validation_split=0.2)

print(f"Evaluation du modèle: {model.evaluate(test_vectorized, test_df['label'])[1]:.3f}")

Epoch 1/4
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 276s 275ms/step - accuracy: 0.5098 - loss: 0.6934 - val_accuracy: 0.5316 - val_loss: 0.7051
Epoch 2/4
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 281s 281ms/step - accuracy: 0.5406 - loss: 0.6781 - val_accuracy: 0.5135 - val_loss: 0.6917
Epoch 3/4
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 273s 273ms/step - accuracy: 0.6809 - loss: 0.5525 - val_accuracy: 0.8487 - val_loss: 0.3742
Epoch 4/4
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 269s 269ms/step - accuracy: 0.8960 - loss: 0.2737 - val_accuracy: 0.8661 - val_loss: 0.3257
313/313 ━━━━━━━━━━━━━━━━━━━━ 22s 69ms/step - accuracy: 0.8687 - loss: 0.3275
Evaluation du modèle: 0.869


### Analyse des embeddings entraînés

Les embeddings générés peuvent ensuite être analysés pour observer les relations sémantiques capturées entre les mots. Nous allons vérifier si les mots similaires partagent des embeddings proches.

In [58]:
# On veut calculer la similarité entre les embeddings de mots.

word_embeddings = embedding_layer.weights[0].numpy()
print(word_embeddings.shape)

# On récupère les embeddings des mots good, great et bad
mot_index_1 = text_vectorization("good").numpy()[0]
mot_index_2 = text_vectorization("great").numpy()[0]
mot_index_3 = text_vectorization("bad").numpy()[0]

embedding_1 = word_embeddings[mot_index_1]
embedding_2 = word_embeddings[mot_index_2]
embedding_3 = word_embeddings[mot_index_3]

#TODO: votre code ici pour calculer la similarité entre les embeddings de good et great, good et bad, great et bad

(20000, 256)


## Word embeddings pré-entraînés

Les embeddings pré-entraînés sont des représentations de mots apprises sur de très grands corpus de texte. Ils capturent les relations sémantiques générales entre les mots. Vous pouvez utiliser des embeddings comme **GloVe**, **Word2Vec**, ou **FastText**, qui sont parmi les plus populaires.

On utilise les embeddings GloVe de dimension 100 --> `glove.6B.100d.txt`
Télécharger les embeddings ici : https://www.kaggle.com/datasets/danielwillgeorge/glove6b100dtxt
Pour plus de détails sur les embeddings GloVe, rendez-vous sur : https://nlp.stanford.edu/projects/glove/

In [59]:
# Pour charger des embeddings pré-entraînés, on peut utiliser la méthode suivante.

embeddings_index = {}
with open(PATH_GLOVE) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


### Comparer des embeddings pré-entraînés

Une fois les embeddings chargés, vous pouvez comparer la similarité entre les vecteurs de mots comme "king", "queen", "paris", "germany", etc., et observer les différences ou les similarités sémantiques entre ces mots.


In [33]:
# Comparer les embeddings de King, Queen, France, Germany, Paris, Berlin etc..

mots = ["king", "queen", "france", "germany", "paris", "berlin", "cat", "dog", "mouse", "lion"]

embeddings = np.array([embeddings_index[word] for word in mots])

#TODO: votre code ici pour calculer la similarité entre les embeddings de ces mots

In [34]:
#TODO: calculer la norme des différences entre les embeddings de France et Germany, Paris et Berlin, France et Cat, France et Paris, Germany et Berlin.
# Qu'observez-vous?

# Votre code ici

## Utilisation des embeddings GloVe

Nous allons maintenant intégrer les embeddings GloVe dans un modèle LSTM pour classer les critiques de films. Les embeddings GloVe sont utilisés pour initialiser la couche d'embedding du modèle LSTM.

In [ ]:
embedding_dim = 100

vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

print(f"word index: {word_index}")

On transforme le vocabulaire du corpus en une matrice d'embeddings, en utilisant les embeddings GloVe.

In [ ]:
embedding_matrix = np.zeros((max_tokens, embedding_dim))

hits, misses = 0, 0
for word, i in word_index.items():
    if i >= max_tokens:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        misses += 1

print(embedding_matrix.shape)
print(f"{hits} mots du vocabulaire trouvés dans GloVe, {misses} non trouvés.")

Créez un modèle LSTM qui intègre la couche d'embedding pré-entraînée avec les embeddings GloVe. Entraînez le modèle sur les données d'entraînement et évaluez-le sur les données de test.

In [ ]:
embedding_layer = keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False
)

model = keras.models.Sequential([
    #TODO: votre code ici.
])

# Compiler, entrainer et évaluer le modèle

#TODO: votre code ici

# Self-attention, à la main

Avant d'utiliser un transformer pré-entraîné comme une boîte noire,
implémentons le mécanisme vu en cours. Pour une phrase de $n$ tokens
représentée par une matrice $X \in \mathbb{R}^{n \times d}$ :

$$ Q = X W^Q, \quad K = X W^K, \quad V = X W^V,
   \qquad \mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V $$

Une dizaine de lignes de numpy suffisent.

In [ ]:
def softmax(x, axis=-1):
    # On soustrait le max avant d'exponentier, pour éviter les overflows.
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def self_attention(X, W_q, W_k, W_v):
    """Self-attention à une tête.

    X   : (n, d)   les n tokens de la phrase
    W_* : (d, d_k) les trois projections apprises

    Doit retourner Y (n, d_k), les tokens contextualisés, et A (n, n), la
    matrice d'attention : A[i, j] = poids accordé au token j quand on
    recalcule i.
    """
    # TODO: votre code ici (4 ou 5 lignes)
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(0)
n, d, d_k = 5, 8, 4

# Des valeurs petites, pour que l'attention de départ soit bien étalée.
X = 0.5 * rng.normal(size=(n, d))
W_q = 0.5 * rng.normal(size=(d, d_k))
W_k = 0.5 * rng.normal(size=(d, d_k))
W_v = 0.5 * rng.normal(size=(d, d_k))

Y, A = self_attention(X, W_q, W_k, W_v)

assert Y.shape == (n, d_k), f"Y devrait être de taille ({n}, {d_k}), pas {Y.shape}"
assert A.shape == (n, n), f"A devrait être de taille ({n}, {n}), pas {A.shape}"
assert np.allclose(A.sum(axis=1), 1.0), "chaque ligne de A doit être une distribution"
print("OK")
print(np.round(A, 2))

#### Questions :

- Que devient la matrice d'attention `A` si l'on multiplie `X` par 100 avant de
  l'appeler ? Pourquoi est-ce un problème, et en quoi la division par
  $\sqrt{d_k}$ aide-t-elle ?
- L'attention est invariante par permutation : vérifiez-le en permutant deux
  lignes de `X`. Quelle conséquence cela a-t-il pour un modèle de langage, et
  comment les transformers s'en sortent-ils ?

# Utilisation d'un modèle Transformer pré-entraîné (BERT)

Pour capturer des relations plus complexes entre les phrases et les mots, nous pouvons utiliser des modèles **Transformer** pré-entraînés, comme **BERT**. Ces modèles génèrent des embeddings de phrases ou de documents, qui peuvent ensuite être utilisés pour la classification.

Utilisez la bibliothèque `SentenceTransformer` pour encoder les phrases en embeddings denses et tester la génération d'embeddings à partir de phrases.

In [ ]:
from sentence_transformers import SentenceTransformer

# On nomme ce modèle `sbert` et non `model` : sinon il serait écrasé par le
# modèle Keras construit juste en dessous, et `sbert.encode` ne marcherait plus
# si vous ré-exécutez la cellule.
sbert = SentenceTransformer("paraphrase-MiniLM-L6-v2")

# Sentences we want to encode. Example:
sentence = ["This framework generates embeddings for each input sentence"]

# Sentences are encoded by calling sbert.encode()
embedding = sbert.encode(sentence)

In [ ]:
embedding

In [ ]:
embedding.shape

Créez un modèle de classification en utilisant un modèle **BERT** pré-entraîné pour encoder les phrases. Vous pouvez utiliser un modèle de classification simple sur les embeddings générés pour la classification, comme une régression logistique.

Encodez les critiques avec `sbert`, puis entraînez un petit classifieur sur ces
embeddings.

**Attention** : encoder les 40 000 critiques d'entraînement prend plusieurs
minutes sur CPU. Commencez par un sous-ensemble (`N_SAMPLES` ci-dessous) pour
vérifier que votre code tourne, puis augmentez.

In [ ]:
# On peut utiliser ces embeddings pour faire de la classification de texte IMDB

N_SAMPLES = 5_000  # augmentez une fois que le code tourne (None = tout le dataset)

train_sub = train_df[:N_SAMPLES] if N_SAMPLES else train_df
test_sub = test_df[:N_SAMPLES] if N_SAMPLES else test_df

train_embeddings = None  # TODO: sbert.encode(..., show_progress_bar=True)
test_embeddings = None   # TODO

# Puis on crée un modèle simple pour la classification de texte.

model = keras.models.Sequential([
    keras.layers.Input(shape=(384,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(train_embeddings, train_sub["label"], epochs=10, batch_size=32, validation_split=0.2)

print(f"Evaluation du modèle: {model.evaluate(test_embeddings, test_sub['label'])[1]:.3f}")

Améliorer maintenant ce modèle en jouant avec les layers etc.

In [ ]:
model = keras.models.Sequential([
    #TODO: votre super modèle ici.
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(train_embeddings, train_sub["label"], epochs=10, batch_size=32, validation_split=0.2)

print(f"Evaluation du modèle: {model.evaluate(test_embeddings, test_sub['label'])[1]:.3f}")
